In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd

from sklearn.ensemble         import RandomForestClassifier, StackingClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.model_selection  import StratifiedKFold
from sklearn.metrics          import roc_auc_score
from xgboost                  import XGBClassifier
from lightgbm                 import LGBMClassifier

from utils.preprocessing        import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils           import get_model_train_eval
from utils.feature_engineering  import drop_highly_correlated_features


In [2]:
# 데이터 로딩 및 기본 전처리 함수
def santander_base_job(columns=['ID'], zcr=0.99, savedDrop=False, isSacled=False, isSplit=False) :
    '''
      SantaderBank CS 분석을 위한 기본 작업
      Arg : 
        columns=['ID'],  # 1차 삭제할 컬럼
        zcr=0.99,        # zero count rate 삭제할 zero_count_rate 기분값
        savedDrop=False, # zcr로 삭제한 컬럼 저장 여부 
        isSacled=False   # scale 여부
        isSplit=False    # isSplit 여부
      Return :   
        isSplit=True  => X_train, X_val, y_train, y_val 
        isSplit=False => X_reduced, y_labels, X_test_reduced
      
      사용예시 : X_reduced, y_labels, X_test_reduced = santander_base_job()
    '''
    # 데이터 로딩 및 기본 전처리
    train, test = load_data()
    X_features, y_labels = split_features_target(train)
    X_test = test.drop(columns=columns, axis=1)

    # zero_count_rate 제거했을때 제거될 컬럼수 149개 잔존
    X_features, X_test = remove_zero_columns2(X_features, X_test, zcr)  
    
    print(X_features.shape)

    # var3 처리
    X_features['var3'] = X_features['var3'].replace(-999999, 2)
    
    # 상관계수 높은 feature들 삭제하기 default 0.95
    X_reduced, to_drop = drop_highly_correlated_features(X_features)
    X_test_reduced = X_test.drop(to_drop, axis=1)  
    print(X_reduced.shape, X_test_reduced.shape)
    
    # 삭제된 컬럼 개수 확인
    print("Train에서 삭제된 컬럼 개수:", len(to_drop))
    
    if savedDrop :
      # for i, col in enumerate(sorted(to_drop), start=1):
      #     print(f"{i:>2} : {col}")      
      # 리스트를 Pandas Series로 변환
      series = pd.Series(sorted(to_drop), name="Dropped_Columns")

      # CSV 파일로 저장
      series.to_csv("../data/99perCorr95DroppedColumns_20251122.xls", index=False)  
      
    if isSplit:           
      if isSacled:
          # 스케일링 
          X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)        
          # 학습/테스트 데이터 분리
          X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)      
      else:
          # 학습/테스트 데이터 분리
          X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)   
          
      return X_train, X_val, y_train, y_val 
    
    else:
      return X_reduced, y_labels, X_test_reduced
#-- EOF -------------------------------------------------------------------------------------------  

In [3]:
X_reduced, y_labels, X_test_reduced = santander_base_job()


Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [9]:
# 모델 앙상블

# 1. Base models 정의
# 최적 하이퍼파라미터: 
# {'max_depth': np.float64(25.0), 
# 'min_samples_leaf': np.float64(1.0), 
# 'min_samples_split': np.float64(7.0), 
# 'n_estimators': np.float64(390.0)}
rf_clf = RandomForestClassifier(
    random_state      = 23,
    n_estimators      = 390,
    max_depth         = 25,
    class_weight      = {0:1, 1:2},
    min_samples_leaf  = 1,
    min_samples_split = 7,
    n_jobs            = -1
)

# 최적 하이퍼파라미터: 
# 'n_estimators': np.float64(320.0), 
# 'colsample_bytree': np.float64(0.8828333771389649), 
# 'gamma': np.float64(0.058408742978283044), 
# 'learning_rate': np.float64(0.12721361071578832), 
# 'max_depth': np.float64(6.0), 
# 'min_child_weight': np.float64(2.0), 
# 'subsample': np.float64(0.845580758889997)}
xgb_clf = XGBClassifier(
    random_state      = 23,    
    n_estimators      = 320,
    colsample_bytree  = 0.88,
    gamma             = 0.058, 
    learning_rate     = 0.13,
    max_depth         = 6,
    scale_pos_weight  = 10,
    min_child_weight  = 2, 
    subsample         = 0.85,    
    eval_metric       ='auc',
    use_label_encoder = False,
    n_jobs            = -1,
)

# 최적 하이퍼파라미터: {
# 'n_estimators': np.float64(660.0), 
# 'colsample_bytree': np.float64(0.7342406115797382), 
# 'learning_rate': np.float64(0.02751826185901235), 
# 'num_leaves': np.float64(42.0), 
# 'reg_alpha': np.float64(0.6127977982911577), 
# 'reg_lambda': np.float64(0.1262561992869149), 
# 'subsample': np.float64(0.973776538266153)}
lgbm_clf = LGBMClassifier(
    random_state     = 23,
    n_estimators     = 300,
    colsample_bytree = 0.73,
    # max_depth      = -1,
    learning_rate    = 0.03,
    num_leaves       = 42,
    reg_alpha        = 0.61,
    reg_lambda       = 0.13,
    subsample        = 0.97,
    class_weight     = {0:1, 1:10},
    n_jobs           = -1,
)

# 2. Meta model 정의
meta_model = LogisticRegression(
    random_state = 23,
    max_iter     = 1000,
    class_weight="balanced"
)

# 3. StackingClassifier 구성
stacking_model = StackingClassifier(
    estimators      = [('rf', rf_clf), ('xgb', xgb_clf), ('lgbm', lgbm_clf)],
    final_estimator = meta_model,
    cv              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs          = -1
)

# 4. 학습 (예시: 전처리된 데이터 X_reduced, y_labels 사용)
stacking_model.fit(X_reduced, y_labels)

# 5. 메타 모델 계수 출력
coef = stacking_model.final_estimator_.coef_[0]
base_models = ['RandomForest', 'XGBoost', 'LightGBM']
for name, weight in zip(base_models, coef):
    print(f"{name} 기여도(계수): {weight:.4f}")

RandomForest 기여도(계수): 2.0190
XGBoost 기여도(계수): -2.1154
LightGBM 기여도(계수): 6.4119


In [13]:
# 2. 스택킹 모델 성능 평가 (전처리된 데이터 X_reduced, y_labels 사용)
from sklearn.metrics import roc_auc_score, f1_score, recall_score

y_pred_proba = stacking_model.predict_proba(X_reduced)[:, 1]
y_pred = stacking_model.predict(X_reduced)

auc_score = roc_auc_score(y_labels, y_pred_proba)
f1 = f1_score(y_labels, y_pred)
recall = recall_score(y_labels, y_pred)

print(f"\nStacking 모델 ROC-AUC: {auc_score:.4f}")
print(f"Stacking 모델 F1-Score: {f1:.4f}")
print(f"Stacking 모델 Recall:   {recall:.4f}")


Stacking 모델 ROC-AUC: 0.9031
Stacking 모델 F1-Score: 0.2722
Stacking 모델 Recall:   0.8384


In [ ]:
#threshold  dropped_features  AUC       F1        Recall
# 0.95      53                0.842169  0.016287  0.008306 (Not Scaled)
# 0.95      53                0.8409    0.0131    0.0066   (Scaled)
# 0.95      57                0.8359    0.0350    0.0183   (Not Scaled)
# 0.95      53                0.8318    0.0218    0.0113   (KFold)

In [14]:
from utils.model_utils import save_model

save_model(stacking_model, 'StackingModel_RF+LGBM+XGB_20251123')

✓ 모델 저장 완료: ../models\StackingModel_RF+LGBM+XGB_20251123.pkl
  파일 크기: 96.75 MB


'../models\\StackingModel_RF+LGBM+XGB_20251123.pkl'